<a href="https://colab.research.google.com/github/Rye-create/Assessment-2/blob/main/Assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install arxiv pymupdf

In [7]:
import arxiv

client = arxiv.Client()
search = arxiv.Search(
    query='all:"laser powder bed fusion"',
    max_results=200,
    sort_by=arxiv.SortCriterion.Relevance,
)

# Only fetch metadata here; no PDFs are downloaded
results = list(client.results(search))
print("Papers found:", len(results))

# Print the first 20 titles for a quick manual check
for r in results[:20]:
    print(r.get_short_id(), r.title)

Papers found: 180
2011.09065v2 Towards Online Monitoring and Data-driven Control: A Study of Segmentation Algorithms for Laser Powder Bed Fusion Processes
2509.03255v1 Anti-scatter grid prototype manufactured via laser powder bed fusion of pure tungsten
2411.18048v1 MeltpoolINR: Predicting temperature field, melt pool geometry, and their rate of change in laser powder bed fusion
2201.09978v1 An empirical model for feedforward control of laser powder bed fusion
2102.03355v1 Thermal Control of Laser Powder Bed Fusion Using Deep Reinforcement Learning
2311.10218v2 Layer-to-Layer Melt Pool Control in Laser Powder Bed Fusion
1902.06785v1 Uncertainty Analysis of Microsegregation during Laser Powder Bed Fusion
2509.01754v1 TransMatch: A Transfer-Learning Framework for Defect Detection in Laser Powder Bed Fusion Additive Manufacturing
2509.16114v1 Real-Time Thermal State Estimation and Forecasting in Laser Powder Bed Fusion
2604.04208v1 Towards Agentic Defect Reasoning: A Graph-Assisted Retrie

In [8]:
import arxiv, pymupdf, requests, os, time

# Remember to add data/raw/ to .gitignore so PDFs are not pushed to GitHub
os.makedirs("data/raw", exist_ok=True)

client = arxiv.Client()
search = arxiv.Search(
    query='all:"laser powder bed fusion" AND all:"AlSi10Mg"',
    max_results=5,
    sort_by=arxiv.SortCriterion.Relevance,
)

for r in client.results(search):
    fname = r.get_short_id().replace("/", "_") + ".pdf"
    path = os.path.join("data/raw", fname)
    try:
        resp = requests.get(r.pdf_url, timeout=60)
        resp.raise_for_status()
        with open(path, "wb") as f:
            f.write(resp.content)
        print(r.title, "->", fname)
    except Exception as e:
        print("Download failed:", r.title, e)
    time.sleep(3)  # Be polite to arXiv: avoid rapid repeated requests

On the influence of powder particle size on single-track formation in laser powder bed fusion of AlSi10Mg alloy -> 2507.23422v1.pdf
Extension of a phase-field KKS model to predict the microstructure evolution in LPBF AlSi10Mg alloy submitted to non isothermal processes -> 2407.11580v1.pdf
Solidification Characteristics of Laser-Powder Bed Fused AlSi10Mg: Role of Building Direction -> 2003.09757v1.pdf
Laser Remelting for Reduced Porosity on Additively Manufactured Aluminium Mirrors -> 2607.21253v1.pdf
Additively manufactured ultra-high vacuum chamber below $10^{-10}$ mbar -> 1903.07708v3.pdf


In [9]:
import re, os, unicodedata, pymupdf

def load_text(path):
    """Read a PDF and return cleaned plain text."""
    doc = pymupdf.open(path)
    text = "\n".join(page.get_text() for page in doc)
    # Normalize ligatures (e.g. the single character "fi") into plain letters
    text = unicodedata.normalize("NFKC", text)
    # Replace the Unicode minus sign with a normal hyphen-minus
    return text.replace("\u2212", "-")

# Keywords that usually appear in papers reporting process parameters
KEYWORDS = ["laser power", "scan speed", "scanning speed", "scan velocity",
            "scanning velocity", "hatch", "layer thickness",
            "relative density", "tensile", "yield strength"]

# Numbers with units are a second signal of experimental content
PATTERNS = {
    "power_W":    r"\b\d+(?:\.\d+)?\s*W\b",
    "speed_mm/s": r"\b\d+(?:\.\d+)?\s*mm\s*/\s*s\b",
    "thick_um":   r"\b\d+(?:\.\d+)?\s*(?:µm|μm|um)\b",
    "MPa":        r"\b\d+(?:\.\d+)?\s*MPa\b",
}

for f in sorted(os.listdir("data/raw")):
    t = load_text(os.path.join("data/raw", f))
    low = t.lower()
    keyword_count = sum(low.count(k) for k in KEYWORDS)
    number_counts = {k: len(re.findall(p, t)) for k, p in PATTERNS.items()}
    print(f, "| total keywords:", keyword_count, "| numeric signals:", number_counts)

1903.07708v3.pdf | total keywords: 2 | numeric signals: {'power_W': 2, 'speed_mm/s': 0, 'thick_um': 7, 'MPa': 0}
2003.09757v1.pdf | total keywords: 9 | numeric signals: {'power_W': 0, 'speed_mm/s': 0, 'thick_um': 11, 'MPa': 0}
2407.11580v1.pdf | total keywords: 7 | numeric signals: {'power_W': 1, 'speed_mm/s': 0, 'thick_um': 1, 'MPa': 0}
2507.23422v1.pdf | total keywords: 52 | numeric signals: {'power_W': 12, 'speed_mm/s': 2, 'thick_um': 55, 'MPa': 0}
2607.21253v1.pdf | total keywords: 7 | numeric signals: {'power_W': 3, 'speed_mm/s': 1, 'thick_um': 14, 'MPa': 2}


In [10]:
import requests
from collections import Counter
from urllib.parse import urlparse

BASE = "https://api.openalex.org/works"
params = {
    "filter": ",".join([
        'title_and_abstract.search:"laser powder bed fusion"',
        "open_access.is_oa:true",
        "best_oa_location.license:cc-by",
        "type:article",
        "from_publication_date:2018-01-01",
    ]),
    "per-page": 200,
    "select": "id,doi,title,publication_year,best_oa_location",
    "mailto": "your_email@example.com",  # Replace with your own email
}

resp = requests.get(BASE, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

print("Total matching works:", data["meta"]["count"])
works = data["results"]

def pdf_host(work):
    """Return the website host of the PDF link, or None if missing."""
    loc = work.get("best_oa_location") or {}
    url = loc.get("pdf_url")
    return urlparse(url).netloc if url else None

# Which websites do the PDF links come from?
host_counts = Counter(pdf_host(w) for w in works)
print("Top PDF hosts:", host_counts.most_common(10))

# Show non-review titles with their PDF host (first 25)
shown = 0
for w in works:
    title = w["title"] or ""
    if "review" in title.lower():
        continue
    print(w["publication_year"], "|", title, "|", pdf_host(w))
    shown += 1
    if shown >= 25:
        break

Total matching works: 4235
Top PDF hosts: [('www.mdpi.com', 53), (None, 52), ('link.springer.com', 31), ('www.sciencedirect.com', 28), ('www.nature.com', 8), ('ars.els-cdn.com', 5), ('onlinelibrary.wiley.com', 5), ('www.tandfonline.com', 3), ('www.frontiersin.org', 3), ('iopscience.iop.org', 2)]
2020 | Processing parameters in laser powder bed fusion metal additive manufacturing | www.sciencedirect.com
2018 | Melt pool temperature and cooling rates in laser powder bed fusion | ars.els-cdn.com
2020 | The role of side-branching in microstructure development in laser powder-bed fusion | www.nature.com
2018 | Laser powder bed fusion in high-pressure atmospheres | link.springer.com
2019 | Dynamics of pore formation during laser powder bed fusion additive manufacturing | www.nature.com
2018 | Laser powder bed fusion at sub-atmospheric pressures | www.sciencedirect.com
2019 | Benchmarking of Laser Powder Bed Fusion Machines | None
2019 | Additive manufacturing of glass with laser powder bed f

In [11]:
import requests, time, csv, os
from urllib.parse import urlparse

BASE = "https://api.openalex.org/works"
EMAIL = "your_email@example.com"  # Replace with your own email

params = {
    "filter": ",".join([
        'title_and_abstract.search:"laser powder bed fusion"',
        "open_access.is_oa:true",
        "best_oa_location.license:cc-by",
        "type:article",
        "from_publication_date:2018-01-01",
    ]),
    "per-page": 200,
    "select": "id,doi,title,publication_year,best_oa_location",
    "cursor": "*",   # Cursor paging lets us walk through all results
    "mailto": EMAIL,
}

rows = []
while True:
    resp = requests.get(BASE, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    for w in data["results"]:
        loc = w.get("best_oa_location") or {}
        pdf_url = loc.get("pdf_url")
        rows.append({
            "openalex_id": w["id"].split("/")[-1],
            "doi": w.get("doi"),
            "title": w.get("title"),
            "year": w.get("publication_year"),
            "license": loc.get("license"),
            "pdf_url": pdf_url,
            "host": urlparse(pdf_url).netloc if pdf_url else None,
        })
    next_cursor = data["meta"].get("next_cursor")
    if not next_cursor or not data["results"]:
        break
    params["cursor"] = next_cursor
    time.sleep(0.3)  # Be polite to the API

print("Total rows collected:", len(rows))

# Keep only hosts where direct PDF download usually works
ALLOWED_HOSTS = {"www.mdpi.com", "link.springer.com", "www.nature.com",
                 "www.frontiersin.org", "pmc.ncbi.nlm.nih.gov"}

candidates = [
    r for r in rows
    if r["host"] in ALLOWED_HOSTS and "review" not in (r["title"] or "").lower()
]
print("Candidates after filtering:", len(candidates))

os.makedirs("data", exist_ok=True)
with open("data/papers_metadata.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(candidates[0].keys()))
    writer.writeheader()
    writer.writerows(candidates)
print("Saved data/papers_metadata.csv")

Total rows collected: 4238
Candidates after filtering: 1479
Saved data/papers_metadata.csv


In [12]:
import requests
from collections import Counter

URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
params = {
    "query": '"laser powder bed fusion" AND OPEN_ACCESS:Y AND HAS_FT:Y',
    "format": "json",
    "resultType": "core",
    "pageSize": 100,
}

resp = requests.get(URL, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

print("Total matching records:", data["hitCount"])

results = data["resultList"]["result"]

# License distribution on the first page
license_counts = Counter(r.get("license") for r in results)
print("License counts (first page):", license_counts)

# Print a few examples for a manual check
for r in results[:15]:
    print(r.get("pubYear"), "|", r.get("pmcid"), "|", r.get("license"),
          "|", (r.get("title") or "")[:80])

Total matching records: 1759
License counts (first page): Counter({'cc by': 85, 'cc by-nc-nd': 15})
2026 | PMC13608174 | cc by | Volumetric Infiltration of Brushite and Monetite Cements into Laser Powder Bed F
2026 | PMC13514805 | cc by | Investigation on the Microstructure and Compressive Properties of W-15Mo Alloy F
2026 | PMC13177226 | cc by-nc-nd | Correction to "Structural Design and Mechanical Properties of Metal Vascular Ste
2026 | PMC13608489 | cc by | Thermal-History-Mediated Defect-Microstructure-Property Evolution in Multi-Laser
2026 | PMC13514143 | cc by | Effect of Cu Particle Cross-Contamination in AlSi10Mg Powder Feedstock: Tensile 
2026 | PMC13467871 | cc by | Fitting-Free Diagnosis of Conduction-Model Breakdown in Laser Powder Bed Fusion.
2026 | PMC13514662 | cc by | Root-Inspired Bio-Interlocking Structure Design and Its Mechanism on Enhancing t
2026 | PMC13164723 | cc by | Powder Spreading Dynamics and Process Optimization at a Heterogeneous Interface 
2026 | PMC1336

In [13]:
import requests, csv, os, time

URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
params = {
    "query": '"laser powder bed fusion" AND OPEN_ACCESS:Y AND HAS_FT:Y',
    "format": "json",
    "resultType": "core",
    "pageSize": 1000,
    "cursorMark": "*",   # Cursor paging walks through all results
}

records = []
while True:
    resp = requests.get(URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    results = data["resultList"]["result"]
    if not results:
        break
    for r in results:
        records.append({
            "pmcid": r.get("pmcid"),
            "doi": r.get("doi"),
            "title": r.get("title"),
            "year": r.get("pubYear"),
            "journal": r.get("journalTitle"),
            "license": r.get("license"),
        })
    next_cursor = data.get("nextCursorMark")
    if not next_cursor or next_cursor == params["cursorMark"]:
        break
    params["cursorMark"] = next_cursor
    time.sleep(0.5)  # Be polite to the API

print("Total records fetched:", len(records))

# Keep only cc by licensed records with a PMCID
EXCLUDE_WORDS = ["correction", "erratum", "editorial", "review",
                 "retraction", "special issue"]

def keep(r):
    title = (r["title"] or "").lower()
    return (
        r["pmcid"]
        and r["license"] == "cc by"
        and not any(w in title for w in EXCLUDE_WORDS)
    )

filtered = [r for r in records if keep(r)]
print("After filtering (cc by, no corrections/editorials/reviews):", len(filtered))

os.makedirs("data", exist_ok=True)
with open("data/epmc_metadata.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(filtered[0].keys()))
    writer.writeheader()
    writer.writerows(filtered)
print("Saved data/epmc_metadata.csv")

Total records fetched: 1759
After filtering (cc by, no corrections/editorials/reviews): 1394
Saved data/epmc_metadata.csv


In [14]:
import requests, csv, os, time

URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
params = {
    "query": '"laser powder bed fusion" AND OPEN_ACCESS:Y AND HAS_FT:Y',
    "format": "json",
    "resultType": "core",
    "pageSize": 100,      # Smaller pages are lighter for the server
    "cursorMark": "*",    # Cursor paging walks through all results
}

def get_with_retry(url, params, max_retries=5):
    """GET request that retries on temporary server errors."""
    wait = 3
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=60)
            if resp.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Temporary error {resp.status_code}")
            resp.raise_for_status()
            return resp
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout) as e:
            print(f"Attempt {attempt}/{max_retries} failed: {e}. Waiting {wait}s...")
            time.sleep(wait)
            wait *= 2
    raise RuntimeError("All retries failed. Try again later.")

records = []
page = 0
while True:
    resp = get_with_retry(URL, params)
    data = resp.json()
    results = data["resultList"]["result"]
    if not results:
        break
    for r in results:
        records.append({
            "pmcid": r.get("pmcid"),
            "doi": r.get("doi"),
            "title": r.get("title"),
            "year": r.get("pubYear"),
            "journal": r.get("journalTitle"),
            "license": r.get("license"),
        })
    page += 1
    print(f"Page {page} done, records so far: {len(records)}")
    next_cursor = data.get("nextCursorMark")
    if not next_cursor or next_cursor == params["cursorMark"]:
        break
    params["cursorMark"] = next_cursor
    time.sleep(1)  # Be polite to the API

print("Total records fetched:", len(records))

# Keep only "cc by" licensed records that have a PMCID
EXCLUDE_WORDS = ["correction", "erratum", "editorial", "review",
                 "retraction", "special issue"]

def keep(r):
    title = (r["title"] or "").lower()
    return (
        r["pmcid"]
        and r["license"] == "cc by"
        and not any(w in title for w in EXCLUDE_WORDS)
    )

filtered = [r for r in records if keep(r)]
print("After filtering (cc by, no corrections/editorials/reviews):", len(filtered))

os.makedirs("data", exist_ok=True)
with open("data/epmc_metadata.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(filtered[0].keys()))
    writer.writeheader()
    writer.writerows(filtered)
print("Saved data/epmc_metadata.csv")

Page 1 done, records so far: 100
Page 2 done, records so far: 200
Page 3 done, records so far: 300
Page 4 done, records so far: 400
Page 5 done, records so far: 500
Page 6 done, records so far: 600
Page 7 done, records so far: 700
Page 8 done, records so far: 800
Page 9 done, records so far: 900
Page 10 done, records so far: 1000
Page 11 done, records so far: 1100
Page 12 done, records so far: 1200
Page 13 done, records so far: 1300
Page 14 done, records so far: 1400
Page 15 done, records so far: 1500
Page 16 done, records so far: 1600
Page 17 done, records so far: 1700
Page 18 done, records so far: 1759
Total records fetched: 1759
After filtering (cc by, no corrections/editorials/reviews): 1394
Saved data/epmc_metadata.csv


In [15]:
import csv, os, time, requests
import xml.etree.ElementTree as ET

os.makedirs("data/raw/xml", exist_ok=True)

with open("data/epmc_metadata.csv", encoding="utf-8") as f:
    papers = list(csv.DictReader(f))

BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest/{}/fullTextXML"
MAX_PAPERS = 20  # Small test first

for p in papers[:MAX_PAPERS]:
    path = os.path.join("data/raw/xml", p["pmcid"] + ".xml")
    if os.path.exists(path):
        continue
    try:
        resp = requests.get(BASE.format(p["pmcid"]), timeout=60)
        resp.raise_for_status()
        with open(path, "wb") as out:
            out.write(resp.content)
    except Exception as e:
        print("FAILED:", p["pmcid"], e)
    time.sleep(1)

# Inspect structure: section titles and number of tables per paper
for fname in sorted(os.listdir("data/raw/xml"))[:5]:
    root = ET.parse(os.path.join("data/raw/xml", fname)).getroot()
    sec_titles = []
    for sec in root.iter("sec"):
        t = sec.find("title")
        if t is not None:
            sec_titles.append("".join(t.itertext()).strip())
    n_tables = len(list(root.iter("table-wrap")))
    print(fname, "| tables:", n_tables)
    print("   sections:", sec_titles[:12])

PMC12494907.xml | tables: 0
   sections: ['Competing interests', 'Footnotes']
PMC12898212.xml | tables: 1
   sections: ['1. Introduction', '2. Materials and Methods', '2.1. Preparation of Experimental Material', '2.2. Analyses', '2.2.1. Microstructure', '2.2.2. Mechanical Properties', '3. Results and Discussion', '3.1. Microstructures', '3.1.1. L-PBF Sample', '3.1.2. Swaged Pieces', '3.2. Mechanical Properties', '3.2.1. Vickers Microhardness']
PMC12985775.xml | tables: 5
   sections: ['1. Introduction', '2. Materials and Methods', '2.1. Powder Feedstock', '2.2. Laser Powder Bed Fusion (LPBF)', '2.3. Heat Treatment', '2.4. Characterization', '3. Results and Discussion', '3.1. Feedstock Characterization', '3.2. Effect of Laser Power on Microstructure', '3.3. Heat Treatment of LPBF-Fabricated Samples', '3.4. Mechanical, Thermal and Electrical Properties', '4. Conclusions']
PMC12985958.xml | tables: 4
   sections: ['1. Introduction', '2. Materials and Methods', '3. Results', '3.1. Microsco

In [16]:
import os, json
import xml.etree.ElementTree as ET

def clean(el):
    """Return the text of an element with whitespace normalized."""
    return " ".join("".join(el.itertext()).split())

def walk(sec, path, out):
    """Recursively collect paragraphs with their section path."""
    t = sec.find("title")
    title = clean(t) if t is not None else ""
    new_path = path + [title] if title else path
    for child in sec:
        if child.tag == "p":
            out.append({"section": " > ".join(new_path), "text": clean(child)})
        elif child.tag == "sec":
            walk(child, new_path, out)

def parse_paper(xml_path):
    root = ET.parse(xml_path).getroot()

    title_el = root.find(".//article-title")
    title = clean(title_el) if title_el is not None else ""

    paragraphs = []
    body = root.find(".//body")
    if body is not None:
        for child in body:
            if child.tag == "sec":
                walk(child, [], paragraphs)
            elif child.tag == "p":
                paragraphs.append({"section": "", "text": clean(child)})

    tables = []
    for tw in root.iter("table-wrap"):
        label = tw.find("label")
        caption = tw.find("caption")
        rows = []
        for tr in tw.iter("tr"):
            cells = [clean(c) for c in tr if c.tag in ("th", "td")]
            if cells:
                rows.append(cells)
        tables.append({
            "label": clean(label) if label is not None else "",
            "caption": clean(caption) if caption is not None else "",
            "rows": rows,
        })

    return {"title": title, "paragraphs": paragraphs, "tables": tables}

KEYWORDS = ["laser power", "scan speed", "scanning speed", "scan velocity",
            "scanning velocity", "hatch", "layer thickness"]

os.makedirs("data/processed", exist_ok=True)

for fname in sorted(os.listdir("data/raw/xml")):
    pmcid = fname.replace(".xml", "")
    paper = parse_paper(os.path.join("data/raw/xml", fname))

    with open(f"data/processed/{pmcid}.json", "w", encoding="utf-8") as f:
        json.dump(paper, f, ensure_ascii=False, indent=2)

    has_methods = any(
        ("method" in p["section"].lower() or "experiment" in p["section"].lower())
        for p in paper["paragraphs"]
    )
    table_text = " ".join(
        " ".join(" ".join(r) for r in t["rows"]) + " " + t["caption"]
        for t in paper["tables"]
    )
    all_text = (" ".join(p["text"] for p in paper["paragraphs"]) + " " + table_text).lower()
    kw_hits = sum(all_text.count(k) for k in KEYWORDS)

    print(pmcid, "| paragraphs:", len(paper["paragraphs"]),
          "| tables:", len(paper["tables"]),
          "| has_methods:", has_methods,
          "| keyword hits:", kw_hits)
    print("   title:", paper["title"][:90])

# Print the first table of one paper to see what a parameter table looks like
sample = json.load(open("data/processed/PMC12985775.json", encoding="utf-8"))
if sample["tables"]:
    t = sample["tables"][0]
    print("\nSample table:", t["label"], "-", t["caption"][:120])
    for row in t["rows"][:8]:
        print("  ", row)

PMC12494907 | paragraphs: 13 | tables: 0 | has_methods: False | keyword hits: 0
   title: Structural integrity of additively manufactured materials
PMC12898212 | paragraphs: 23 | tables: 1 | has_methods: True | keyword hits: 7
   title: Enhancing the Mechanical Performance of Laser Powder Bed Fusion Prepared 316L Stainless St
PMC12985775 | paragraphs: 73 | tables: 5 | has_methods: True | keyword hits: 40
   title: Heat Treatment Effects on Laser Powder Bed Fused CuNi2.5SiCr Alloy: Microstructure, Hardne
PMC12985958 | paragraphs: 36 | tables: 4 | has_methods: True | keyword hits: 12
   title: Influence of Base Plate Preheating on Laser Powder Bed Fusion–Processed EN AW-7075 Alumini
PMC13027643 | paragraphs: 37 | tables: 3 | has_methods: True | keyword hits: 8
   title: Microstructure and Mechanical Properties of 1080 Plain Carbon Steel Fabricated by Laser Po
PMC13028084 | paragraphs: 40 | tables: 8 | has_methods: True | keyword hits: 4
   title: Studies of the Thermophysical Properties 